# YOLO PyTorch to TensorFlow Lite Converter

This notebook converts your YOLO `.pt` model to `.tflite` format for mobile deployment.

**Runtime:** Make sure to use GPU runtime (Runtime → Change runtime type → GPU)


## Step 1: Upload Your Model


In [ ]:
from google.colab import files
uploaded = files.upload()

# Find the uploaded .pt file
import os
pt_files = [f for f in os.listdir('.') if f.endswith('.pt')]
if pt_files:
    MODEL_PATH = pt_files[0]
    print(f"✓ Found model: {MODEL_PATH}")
else:
    print("❌ No .pt file found. Please upload your model.")
    MODEL_PATH = None


## Step 2: Install Dependencies


In [ ]:
!pip install ultralytics tensorflow onnx onnx-tf tf2onnx -q
print("✓ Dependencies installed")


## Step 3: Convert Model


In [ ]:
import torch
import tensorflow as tf
from ultralytics import YOLO
import numpy as np
from pathlib import Path
import onnx
from onnx_tf.backend import prepare

print("=" * 60)
print("YOLO PyTorch to TensorFlow Lite Converter")
print("=" * 60)
print()

# Configuration
INPUT_SIZE = 640  # YOLO input size
OUTPUT_PATH = "best.tflite"

if MODEL_PATH is None:
    print("❌ Please upload your model first")
else:
    # Step 1: Load YOLO model
    print("📥 Step 1: Loading YOLO model...")
    model = YOLO(MODEL_PATH)
    print(f"✓ Model loaded: {MODEL_PATH}")

    # Step 2: Export to ONNX
    print("\n📤 Step 2: Exporting to ONNX...")
    onnx_path = MODEL_PATH.replace('.pt', '.onnx')
    model.export(format='onnx', imgsz=INPUT_SIZE, simplify=True, opset=13)
    print(f"✓ ONNX model saved: {onnx_path}")

    # Step 3: Convert ONNX to TensorFlow
    print("\n🔄 Step 3: Converting ONNX to TensorFlow...")
    onnx_model = onnx.load(onnx_path)
    tf_rep = prepare(onnx_model)
    tf_model_path = onnx_path.replace('.onnx', '_tf')
    tf_rep.export_graph(tf_model_path)
    print(f"✓ TensorFlow model saved: {tf_model_path}")

    # Step 4: Convert to TFLite
    print("\n🔧 Step 4: Converting to TensorFlow Lite...")
    converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_path)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float32]
    converter.inference_input_type = tf.float32
    converter.inference_output_type = tf.float32

    tflite_model = converter.convert()

    # Save
    with open(OUTPUT_PATH, 'wb') as f:
        f.write(tflite_model)

    print(f"✓ TensorFlow Lite model saved: {OUTPUT_PATH}")
    print(f"   File size: {len(tflite_model) / (1024 * 1024):.2f} MB")

    # Step 5: Verify
    print("\n✅ Step 5: Verifying TFLite model...")
    interpreter = tf.lite.Interpreter(model_path=OUTPUT_PATH)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print(f"   Input shape: {input_details[0]['shape']}")
    print(f"   Input type: {input_details[0]['dtype']}")
    print(f"   Output shape: {output_details[0]['shape']}")
    print(f"   Output type: {output_details[0]['dtype']}")

    # Test
    print("\n🧪 Step 6: Testing with dummy input...")
    test_input = np.random.rand(1, 3, INPUT_SIZE, INPUT_SIZE).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], test_input)
    interpreter.invoke()
    test_output = interpreter.get_tensor(output_details[0]['index'])
    print(f"   Test output shape: {test_output.shape}")
    print(f"   ✓ Test successful!")

    print("\n" + "=" * 60)
    print("✅ Conversion completed successfully!")
    print("=" * 60)


## Step 4: Download the Model


In [ ]:
from google.colab import files
files.download('best.tflite')
print("✓ Model downloaded!")
